# 01 — Diagnostics & Feature Prototyping

Two purposes:

1. **Understand what `lightgbm_kaggle_v1` actually learned** — feature importance, rolling IC, regime conditioning. The current scoreboard says it works (IC 0.007, t-stat 2.39); this notebook tells us *why*.
2. **Prototype new features** in-notebook before adding them to `src/price_model/features/`. Borrowed from the Kaggle hedge-fund notebooks: EWM, pairwise interactions, cross-sectional min/max distance.

Workflow: explore → prototype → if a feature looks promising, lift it into `cross_features.py` and run a full walk-forward via `python -m price_model.cli run --experiment ...`.

Run with: `jupyter lab notebooks/01_diagnostics.ipynb`

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from price_model.data.loaders import load_panel
from price_model.features.pipeline import build_feature_matrix, drop_warmup_rows
from price_model.features.targets import add_forward_excess_return
from price_model.serving.store import PredictionStore
from price_model.eval.metrics import compare_models
from price_model.models.boosting import LightGBMModel

pl.Config(tbl_width_chars=200, tbl_cols=20)
plt.rcParams['figure.figsize'] = (12, 4)

store = PredictionStore()
print('Store path:', store.path)

## 1. Feature importance — `lightgbm_kaggle_v1`

Loads the saved model artifact and prints the gain-based importance averaged across all 5 seeds.

**What to look for:**
- If 1–2 features dominate, the rest are mostly noise → prune them or fold them into interactions.
- If importance is spread evenly, the model is averaging weak independent signals → ensemble logic, add more independent signals.
- If `return_5d` dominates, the model is basically the LastReturn baseline with sign correction. The new features aren't pulling their weight.

In [ ]:
from price_model.models.base import ModelConfig

feats_kaggle = [
    'return_5d', 'momentum_60', 'vol_20', 'rsi_14', 'distance_ma_200',
    'momentum_60_sector_rel', 'idio_vol_20',
    'momentum_60_rank', 'vol_20_rank', 'distance_ma_200_rank',
]

# Load + build features (cached after first call)
_panel_imp = load_panel(universe='sp500', start='2017-01-01')
_matrix_imp = build_feature_matrix(_panel_imp, feature_names=feats_kaggle, target_horizon=5)
_matrix_imp = drop_warmup_rows(_matrix_imp, feats_kaggle).drop_nulls('y')
print(f'Matrix for importance fit: {_matrix_imp.height:,} rows after warmup')

# Single seed of the Kaggle config — importance ranking is stable across seeds.
cfg = ModelConfig(
    model_id='lightgbm_kaggle_importance',
    feature_cols=tuple(feats_kaggle),
    params={
        'learning_rate': 0.02, 'num_leaves': 128, 'min_data_in_leaf': 200,
        'feature_fraction': 0.7, 'bagging_fraction': 0.7, 'bagging_freq': 5,
        'lambda_l1': 0.1, 'lambda_l2': 5.0, 'n_estimators': 800,
        'val_fraction': 0.1, 'early_stopping_rounds': 100,
        'seeds': [42],
    },
)
model = LightGBMModel(cfg)
model.fit(_matrix_imp)

imp = pd.Series(model.feature_importance()).sort_values(ascending=False)
imp_pct = (imp / imp.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
imp_pct.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Gain % of total')
ax.set_title('Kaggle-config LightGBM — gain-based importance (1 seed, full panel)')
plt.tight_layout(); plt.show()

print(imp_pct.to_string())

## 2. Realized returns — load panel, attach target

Needed for everything that follows. Cached on first run.

In [ ]:
panel = load_panel(universe='sp500', start='2017-01-01')
panel = add_forward_excess_return(panel, horizon_days=5)
realized = panel.select('date', 'ticker', pl.col('y').alias('realized'))
print(f'Panel: {panel.height:,} rows, {panel["ticker"].n_unique()} tickers, '
      f'{panel["date"].min()} → {panel["date"].max()}')

## 3. Full untruncated comparison table

Side-by-side of every LightGBM variant currently in the store. The single place to check whether new experiments earned their keep.

In [ ]:
preds = store.query(
    'SELECT model_id, prediction_date AS date, ticker, prediction FROM predictions'
)
joined = preds.join(realized, on=['date', 'ticker'], how='inner')
summary = compare_models(joined, horizon_days=5).sort('information_coefficient', descending=True)
with pl.Config(tbl_width_chars=300, tbl_cols=20):
    print(summary)

## 4. Rolling 60-day IC

Same average IC can come from "stable +0.005 every month" or "alternating ±0.05." Plot the rolling per-month IC for each model. If `lightgbm_kaggle_v1`'s line is consistently above zero with small variance, the t-stat 2.39 is a *real* sustained edge. If it's swinging hard, the average is masking instability.

In [ ]:
from scipy.stats import spearmanr

def per_date_ic(df: pl.DataFrame) -> pl.DataFrame:
    """Compute Spearman IC per (model_id, date)."""
    rows = []
    for (model_id, d), grp in df.group_by(['model_id', 'date']):
        sub = grp.drop_nulls(['prediction', 'realized'])
        if sub.height < 5:
            continue
        rho, _ = spearmanr(sub['prediction'].to_numpy(), sub['realized'].to_numpy())
        if rho is not None and np.isfinite(rho):
            rows.append({'model_id': model_id, 'date': d, 'ic': float(rho)})
    return pl.DataFrame(rows)

ic_by_date = per_date_ic(joined).sort('date')

# Rolling 60-day mean IC per model, plotted
ic_pd = ic_by_date.to_pandas().set_index('date').sort_index()
fig, ax = plt.subplots(figsize=(12, 4))
for mid, g in ic_pd.groupby('model_id'):
    g['ic'].rolling('60D').mean().plot(ax=ax, label=mid, alpha=0.85)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_title('Rolling 60-day mean IC by model')
ax.legend(loc='best')
plt.tight_layout(); plt.show()

## 5. Regime-conditional IC (high vol vs low vol)

Use the cross-sectional std of the universe's daily returns as a regime proxy. High-vol days are the ones we most want predictions to *not* fall apart on. Split the IC time series at the median and compare.

In [ ]:
# Universe-wide cross-sectional std of daily log returns per date
daily_ret = panel.sort(['ticker', 'date']).with_columns(
    (pl.col('adj_close').log() - pl.col('adj_close').log().shift(1)).over('ticker').alias('log_ret')
)
regime = (
    daily_ret.drop_nulls('log_ret')
    .group_by('date')
    .agg(pl.col('log_ret').std().alias('xs_vol'))
    .sort('date')
)
vol_median = float(regime['xs_vol'].median())
print(f'Cross-sectional vol median across all dates: {vol_median:.4f}')

joined_w_regime = (
    ic_by_date.join(regime, on='date', how='left')
    .with_columns(
        (pl.col('xs_vol') > vol_median).alias('high_vol_day')
    )
    .drop_nulls('xs_vol')
)

regime_summary = joined_w_regime.group_by(['model_id', 'high_vol_day']).agg(
    pl.col('ic').mean().alias('mean_ic'),
    pl.col('ic').std().alias('std_ic'),
    pl.col('ic').count().alias('n'),
).sort(['model_id', 'high_vol_day'])
with pl.Config(tbl_width_chars=200):
    print(regime_summary)

## 6. Prototype new features

Same pattern as `src/price_model/features/`: compute → cross-sectional normalize → eyeball signal vs. forward return.

Build them on the panel here. If a feature shows non-trivial correlation with realized returns, lift it into `cross_features.py` and add to an experiment YAML to test under the full walk-forward harness.

In [ ]:
from scipy.stats import spearmanr as _spearmanr

def quick_ic(panel_with_feat: pl.DataFrame, feat_col: str) -> dict:
    """Per-date Spearman IC of a feature vs. realized forward excess return ('y').
    Returns avg IC and t-stat. NOT a backtest — just a quick check whether the
    feature has any predictive signal at all."""
    df = panel_with_feat.drop_nulls([feat_col, 'y'])
    rows = []
    for d, grp in df.group_by('date'):
        if grp.height < 10:
            continue
        rho, _ = _spearmanr(grp[feat_col].to_numpy(), grp['y'].to_numpy())
        if rho is not None and np.isfinite(rho):
            rows.append(rho)
    if not rows:
        return {'feat': feat_col, 'mean_ic': np.nan, 't_stat': np.nan, 'n_dates': 0}
    arr = np.array(rows)
    return {
        'feat': feat_col,
        'mean_ic': float(arr.mean()),
        't_stat': float(arr.mean() * np.sqrt(len(arr)) / arr.std(ddof=1)),
        'n_dates': len(arr),
    }

# Base panel for feature prototyping: panel + 5-day forward excess return
proto = panel.sort(['ticker', 'date'])
print(f'Proto panel: {proto.height:,} rows, {proto["ticker"].n_unique()} tickers')

### 6a. EWM (exponentially weighted) variants

Weights recent observations more heavily than a flat window. Used by all four ML notebooks from Kaggle. Cheap to compute.

In [ ]:
# Polars doesn't have first-class EWM, so use pandas via .to_pandas() for the prototype.
# If a variant wins, port to a polars expression in cross_features.py.

p = proto.to_pandas().sort_values(['ticker', 'date'])
p['log_ret'] = p.groupby('ticker')['adj_close'].transform(lambda x: np.log(x).diff())

p['ret_ewm_5']  = p.groupby('ticker')['log_ret'].transform(lambda x: x.ewm(span=5, adjust=False).mean())
p['ret_ewm_10'] = p.groupby('ticker')['log_ret'].transform(lambda x: x.ewm(span=10, adjust=False).mean())
p['ret_ewm_20'] = p.groupby('ticker')['log_ret'].transform(lambda x: x.ewm(span=20, adjust=False).mean())
p['vol_ewm_20'] = p.groupby('ticker')['log_ret'].transform(lambda x: x.ewm(span=20, adjust=False).std())

back = pl.from_pandas(p)

rows = []
for f in ['ret_ewm_5', 'ret_ewm_10', 'ret_ewm_20', 'vol_ewm_20']:
    rows.append(quick_ic(back, f))
print(pl.DataFrame(rows))

### 6b. Pairwise interactions of base features

Kaggle notebooks spread/ratio their top features: `f_a - f_b`, `f_a / (f_b + eps)`. For us, the candidates we'd expect to matter are:

- `return_5d / vol_20` — Sharpe-ish recency signal
- `momentum_60 / vol_20` — risk-adjusted momentum
- `momentum_60 - distance_ma_200` — short-term vs long-term trend disagreement

We need to build the base features first (using the existing pipeline) before we can prototype interactions on top.

In [ ]:
feats = ['return_5d', 'momentum_60', 'vol_20', 'distance_ma_200']
matrix = build_feature_matrix(panel, feature_names=feats, target_horizon=5)
matrix = drop_warmup_rows(matrix, feats)

EPS = 1e-7
matrix = matrix.with_columns(
    (pl.col('return_5d') / (pl.col('vol_20').abs() + EPS)).alias('ret5_over_vol20'),
    (pl.col('momentum_60') / (pl.col('vol_20').abs() + EPS)).alias('mom60_over_vol20'),
    (pl.col('momentum_60') - pl.col('distance_ma_200')).alias('mom60_minus_distMA200'),
    (pl.col('return_5d') * pl.col('momentum_60')).alias('ret5_x_mom60'),
)

rows = []
for f in ['ret5_over_vol20', 'mom60_over_vol20', 'mom60_minus_distMA200', 'ret5_x_mom60']:
    rows.append(quick_ic(matrix, f))
print(pl.DataFrame(rows))

### 6c. Cross-sectional min/max distance

For a feature `x`, compute `x - min(x on this date)` and `max(x on this date) - x`. Robust to outliers in a way z-score isn't, and gives the tree more flexible cut points than the rank features we already have. Test on `momentum_60` and `vol_20`.

In [ ]:
matrix = matrix.with_columns(
    (pl.col('momentum_60') - pl.col('momentum_60').min().over('date')).alias('mom60_dist_min'),
    (pl.col('momentum_60').max().over('date') - pl.col('momentum_60')).alias('mom60_dist_max'),
    (pl.col('vol_20') - pl.col('vol_20').min().over('date')).alias('vol20_dist_min'),
    (pl.col('vol_20').max().over('date') - pl.col('vol_20')).alias('vol20_dist_max'),
)

rows = []
for f in ['mom60_dist_min', 'mom60_dist_max', 'vol20_dist_min', 'vol20_dist_max']:
    rows.append(quick_ic(matrix, f))
print(pl.DataFrame(rows))

In [ ]:


panel = load_panel(universe="sp500_pit", start="2017-01-01", pit_filter=True)
matrix = build_feature_matrix(
    panel,
    feature_names=["momentum_12_1", "distance_52w_high", "return_1d"],
    normalize_kind="zscore",
    target_horizon=5,
)
matrix = drop_warmup_rows(matrix, ["momentum_12_1", "distance_52w_high", "return_1d"])

for feat in ["momentum_12_1", "distance_52w_high", "return_1d"]:
    print(feat, "univariate IC + t-stat:", quick_ic(matrix, feat))

## 7. Decision rule for promoting a prototype

A feature is worth lifting into `cross_features.py` and testing under the full walk-forward harness if:

- `|mean_ic| ≥ 0.005` *and* `|t_stat| ≥ 2.0` — meaningfully different from zero across all dates
- OR the sign is opposite to what you'd intuitively expect (e.g., short-term reversal: positive `return_5d` → negative forward return) — that's still useful, just flip its sign downstream

Features that hit `|t_stat| < 1` are noise — don't bother.

**Important caveat:** quick_ic above is *raw univariate* signal. A feature can have weak univariate signal but contribute via interaction with other features in a tree model. So don't kill a feature solely on weak quick_ic — but do prioritize the ones that pass the bar.

Once you decide which to keep:

1. Implement the feature class in `src/price_model/features/cross_features.py` (decorated with `@register`).
2. The no-leakage parametrized test auto-picks it up — run `pytest tests/test_no_leakage.py -v` to confirm.
3. Create `config/experiments/extended_kaggle_v2.yaml` with the new feature list.
4. `python -m price_model.cli run --experiment extended_kaggle_v2`
5. Re-run section 3 above to see the new model in the comparison table.